# M5 Verification: Coordination Evidence

This notebook verifies the deterministic M5 replacement for the legacy constant LLM score. M5 is a global transcript-corpus measure by temporal marker, not a team-semester or session-level outcome.

In [ ]:
from pathlib import Path
import json
import sys

import pandas as pd
import plotly.express as px
from IPython.display import Image, display

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'paper_v9').is_dir():
    ROOT = ROOT.parent
METRICS = ROOT / 'paper_v9' / 'data' / 'metrics'
FIGURES = ROOT / 'paper_v9' / 'figures'
FIGURES.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(ROOT))
density = pd.read_csv(METRICS / 'm5_marker_density.csv')
composition = pd.read_csv(METRICS / 'm5_marker_composition.csv')
coverage = pd.read_csv(METRICS / 'm5_corpus_coverage.csv')
audit = pd.read_csv(METRICS / 'm5_evidence_audit_trail.csv')
lexicon = json.loads((METRICS / 'm5_lexicon.json').read_text(encoding='utf-8'))
metadata = json.loads((METRICS / 'm5_coordination_evidence.metadata.json').read_text(encoding='utf-8'))
legacy = pd.read_csv(ROOT / 'paper_v8' / 'data' / 'm5_coordination_friction_trajectory.csv')


## 1. Verification contract

The verification contract requires RQ2 linkage, a global corpus grain, no new LLM calls, explicit observed-semester coverage, and no silent imputation.

In [ ]:
paper_text = (ROOT / 'paper_v8' / 'latex_code' / 'main.tex').read_text(encoding='utf-8')
m5_position = paper_text.index(r'\textbf{M5 --')
rq2_position = paper_text.index(r'\subsubsection{RQ2:')
rq3_position = paper_text.index(r'\subsubsection{RQ3:')
assert rq2_position < m5_position < rq3_position
assert metadata['rq'] == 'RQ2'
assert metadata['unit_of_analysis'] == 'global transcript corpus by temporal marker'
assert metadata['llm_calls_required'] is False
assert metadata['inference'] == 'descriptive_only'
assert metadata['observed_semesters'] == ['2025.2']
assert density['temporal_marker'].tolist() == ['T1', 'T2', 'T3']
assert density['measurement_status'].eq('available').all()
assert coverage['observed_semesters'].astype(str).eq('2025.2').all()
print('M5 RQ2, grain, LLM, coverage, and inference contract: PASS')

## 2. V8 to V9 traceability

The legacy artifact contains one constant score per marker. The V9 decision replaces that opaque score with auditable density, composition, coverage, and source-chunk evidence.

In [ ]:
traceability = pd.DataFrame([
    {'v8_recommendation': 'Replace the constant LLM score with a changing qualitative trajectory.', 'v9_decision': 'Compute deterministic lexical marker density per 1,000 corpus tokens.', 'status': 'applied', 'evidence': 'm5_marker_density.csv', 'limitation_or_approval': 'Candidate textual evidence; no causal interpretation.'},
    {'v8_recommendation': 'Separate alignment, handoff, integration, blocker, and rework evidence.', 'v9_decision': 'Publish long-form subtype counts and shares by temporal marker.', 'status': 'applied', 'evidence': 'm5_marker_composition.csv and m5_lexicon.json', 'limitation_or_approval': 'Regex matches require human qualitative review.'},
    {'v8_recommendation': 'Declare corpus and session/chunk coverage.', 'v9_decision': 'Publish chunk, source-session, character, token, marker, and semester coverage.', 'status': 'applied', 'evidence': 'm5_corpus_coverage.csv and metadata', 'limitation_or_approval': 'Only 2025.2 is observed; no semester is imputed.'},
    {'v8_recommendation': 'Preserve evidence for audit rather than treating chunks as observations.', 'v9_decision': 'Publish a source-chunk queue with provenance and literal text.', 'status': 'applied', 'evidence': 'm5_evidence_audit_trail.csv', 'limitation_or_approval': 'Audit queue is not a statistical sample.'},
])
assert set(traceability['status']) == {'applied'}
display(traceability)
display(pd.DataFrame([{'legacy_rows': len(legacy), 'legacy_distinct_scores': legacy['coordination_friction'].nunique(), 'legacy_score_range': legacy['coordination_friction'].max() - legacy['coordination_friction'].min(), 'lexicon_version': lexicon['version']}]))

In [ ]:
subtypes = ['alignment', 'handoff', 'integration', 'blocker', 'rework']
assert len(density) == 3
assert density['transcript_chunk_n'].tolist() == [35, 37, 28]
assert density['source_session_n'].eq(2).all()
assert density['token_n'].gt(0).all()
assert density['friction_marker_n'].ge(0).all()
assert density['friction_marker_density_per_1k_tokens'].notna().all()
assert set(composition['friction_subtype']) == set(subtypes)
assert composition.groupby('temporal_marker')['marker_n'].sum().equals(density.set_index('temporal_marker')['friction_marker_n'])
assert len(audit) > 0
assert audit['transcript_text'].str.strip().ne('').all()
assert audit['lexicon_version'].eq(lexicon['version']).all()
print('M5 schemas, ranges, composition conservation, coverage, and audit evidence: PASS')

## 3. Article-ready artifacts

The plots below are derived from official M5 CSV outputs. They contrast density and subtype composition descriptively; they do not align M5 with team-level M3/M4 observations or imply correlation.

In [ ]:
density_figure = px.line(density, x='temporal_marker', y='friction_marker_density_per_1k_tokens', markers=True, title='M5 friction-marker density by temporal marker')
density_figure.update_yaxes(title='Candidate markers per 1,000 corpus tokens')
composition_figure = px.bar(composition, x='temporal_marker', y='marker_share', color='friction_subtype', barmode='stack', title='M5 candidate friction composition')
composition_figure.update_yaxes(title='Share of candidate markers')
for figure, stem in ((density_figure, 'm5_marker_density'), (composition_figure, 'm5_marker_composition')):
    figure.write_html(METRICS / f'{stem}.html', include_plotlyjs='cdn')
    for extension in ('pdf', 'svg', 'png'):
        figure.write_image(FIGURES / f'{stem}.{extension}', scale=2 if extension == 'png' else 1)
for stem in ('m5_marker_density', 'm5_marker_composition'):
    assert all((FIGURES / f'{stem}.{extension}').is_file() and (FIGURES / f'{stem}.{extension}').stat().st_size > 0 for extension in ('pdf', 'svg', 'png'))
    assert (METRICS / f'{stem}.html').is_file() and (METRICS / f'{stem}.html').stat().st_size > 0
print('M5 HTML, PDF, SVG, and PNG figures generated: PASS')

In [ ]:
display(density[['temporal_marker', 'transcript_chunk_n', 'source_session_n', 'token_n', 'friction_marker_n', 'friction_marker_density_per_1k_tokens']])
display(composition.sort_values(['temporal_marker', 'marker_n'], ascending=[True, False]))
display(coverage)
display(audit.head(5)[['temporal_marker', 'session_id', 'friction_marker_n', 'transcript_text']])
display(Image(filename=str(FIGURES / 'm5_marker_density.png'), width=900))
display(Image(filename=str(FIGURES / 'm5_marker_composition.png'), width=900))
print('M5 artifact demo: official tables, audit evidence, and figures displayed')

## Preliminary RQ2 analysis

M5 contributes a descriptive qualitative axis to RQ2: it reports how candidate coordination-friction language is distributed across the three observed temporal markers in one global transcript corpus. The result cannot be interpreted as team performance, effort, productivity, quality, a causal mechanism, or a correlation with M3/M4. Only semester `2025.2` has transcript coverage; absent semesters remain unavailable rather than imputed.

In [ ]:
summary = density[['temporal_marker', 'friction_marker_density_per_1k_tokens', 'friction_marker_n', 'token_n']].copy()
summary['semester_coverage'] = ', '.join(metadata['observed_semesters'])
summary['analysis_level'] = metadata['unit_of_analysis']
summary['inference'] = metadata['inference']
assert summary['analysis_level'].eq('global transcript corpus by temporal marker').all()
assert summary['inference'].eq('descriptive_only').all()
display(summary)
print('Preliminary RQ2 reading: M5 supplies a temporal textual contrast only; no association or causal claim is supported.')